Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

Read from bronze table

In [0]:
df=spark.table("workspace.bronze.cust_az12")

Silver Transformations

Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(col(field.name)))


Birthdate Validation

In [0]:
df=df.withColumn("bdate",
                 F.when(col("bdate")>F.current_date(),None)
                 .otherwise(col("bdate")))

Gender Normalization

In [0]:
df=df.withColumn(
    "gender",
    F.when(F.upper(col("gen")).isin("M","MALE"),"Male")
.when(F.upper(col("gen")).isin("F","FEMALE"),"Female")
)

Renaming Columns

In [0]:
RENAME_MAP={
    "cid":"customer_number",
    "bdate":"birth_date",
}
for old,new in RENAME_MAP.items():
    df=df.withColumnRenamed(old,new)

Sanity checks of dataframe

In [0]:
df.limit(10).display()

Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer")